# Notebook 6 of 7: Improving Generation

## Why AI Repeats Itself (and How to Fix It)

**Series: Understanding AI Through Italian Music**

---

You have seen it in the previous notebooks: **"sole sole sole sole sole..."** or **"volere volere volere..."** -- the same words cycling endlessly. This is called **repetition collapse**, and it is one of the most common problems in AI text generation.

In this notebook, we will understand **WHY** it happens and learn the techniques that fix it. These same techniques are used in ChatGPT, Claude, and every modern AI system.

Here is the basic problem: when we always pick the MOST likely next word, the model gets stuck in a loop. If "sole" is the most likely word after "sole," we get "sole sole sole" forever. The solution is to introduce some controlled randomness -- not so much that the output becomes gibberish, but enough that the model can explore different paths.

We will learn four strategies:

1. **Temperature** -- a "confidence dial" that controls how adventurous the model is
2. **Top-k Sampling** -- only consider the k most likely next words
3. **Top-p (Nucleus) Sampling** -- a smarter version that adapts to context
4. **Repetition Penalty** -- directly discourage the model from repeating itself

By the end, you will understand exactly what happens behind the scenes when you use any modern AI tool.

## Setup

Run the cell below to load all the tools we need. This brings in our models (RNN, LSTM, Transformer), the training functions, the generation functions, and the visualization helpers.

You do not need to understand every import -- just run it and move on.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
import matplotlib.pyplot as plt
import numpy as np

# Our custom modules
from src.dataset import ItalianLyricsDataset, load_lyrics
from src.models import RNNModel, LSTMModel, create_transformer_model
from src.training import train_model, get_optimizer
from src.generation import (
    generate_rnn_lstm,
    generate_transformer,
    generate_greedy,
    top_k_top_p_filtering,
)
from src.visualization import plot_next_word_probabilities, display_generation_comparison

# Show charts inside the notebook
%matplotlib inline

# Device setup
device = torch.device('cpu')

print("All tools loaded successfully!")

## The Repetition Problem: A Demonstration

Before we can fix the problem, we need to see it clearly. Let's train a quick RNN model on 300 songs for just 2 epochs -- enough for it to learn some Italian patterns -- and then ask it to generate lyrics using the simplest possible strategy: **always pick the most likely next word.**

This strategy is called **greedy decoding**. It sounds like a good idea (why not always pick the best word?) but as you will see, it leads to a very predictable failure.

When we always pick the MOST likely next word, the model gets stuck in a loop. If "sole" is the most likely word after "sole", we get "sole sole sole" forever.

In [ ]:
# --- Step 1: Load data and prepare for training ---
lyrics = load_lyrics('../data/italian_lyrics.txt', max_songs=300)
print(f"Loaded {len(lyrics)} songs for training")

# Set up the tokenizer (converts words to numbers and back)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Create dataset and dataloader
dataset = ItalianLyricsDataset(lyrics, tokenizer, max_length=128)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# --- Step 2: Build and train a quick RNN ---
VOCAB_SIZE = len(tokenizer)
rnn_model = RNNModel(vocab_size=VOCAB_SIZE, embedding_dim=256, hidden_dim=512).to(device)

optimizer = get_optimizer(rnn_model, is_transformer=False)

print("\nTraining RNN (this will take about 1-2 minutes)...")
rnn_history = train_model(
    rnn_model, dataloader, optimizer, device,
    epochs=2, is_transformer=False, model_name="RNN"
)

print("\nTraining complete!")

Now let's generate lyrics using **greedy decoding** -- always picking the single most likely next word. Watch what happens:

In [ ]:
# Generate with greedy decoding -- always pick the most likely word
seed_texts = ["Amore mio", "La vita", "Sotto il sole"]

print("GREEDY DECODING -- always pick the most likely next word:")
print("=" * 60)
for seed in seed_texts:
    greedy_output = generate_greedy(rnn_model, tokenizer, seed, max_length=50)
    print(f"\nSeed: \"{seed}\"")
    print(f"Output: {greedy_output}")
    print("-" * 60)

print("\nDo you see the repetition? The same words appear over and")
print("over. This is 'repetition collapse' -- the model is stuck")
print("in a loop because it always picks the same 'best' word.")

## Understanding Probabilities: How the Model "Thinks"

Here is the key insight: **the model does not pick ONE word.** At every step, it assigns a probability to **every single word in its vocabulary** -- all 50,257 of them. Some words get a high probability (they are likely next words), and most get a probability very close to zero (they would make no sense in that position).

Greedy decoding throws away all that rich information and just picks the single top word. But what if the top word has a 15% chance, and the second-best word has a 13% chance? They are almost equally good! By always picking the top word, we miss out on perfectly reasonable alternatives.

The model doesn't pick ONE word -- it assigns a probability to EVERY word in its vocabulary. Generation is about choosing from these probabilities.

Let's look inside the model and see these probabilities for ourselves.

In [ ]:
# Let's look inside the model's "mind" for a specific seed text
seed_text = "Amore mio"

# Convert seed text to numbers (tokens)
input_ids = tokenizer.encode(seed_text, return_tensors='pt').to(device)

# Ask the model: what word comes next?
rnn_model.eval()
with torch.no_grad():
    outputs = rnn_model(input_ids)
    # Get the predictions for the NEXT word (after the last input word)
    next_word_logits = outputs[0, -1, :]

# Convert raw scores ("logits") to probabilities using softmax
# Softmax squashes all values so they add up to 1.0 (100%)
probabilities = F.softmax(next_word_logits, dim=-1)

# Get the top 10 most likely next words
top_probs, top_indices = torch.topk(probabilities, 10)
top_words = [tokenizer.decode([idx]) for idx in top_indices]
top_probs_list = top_probs.tolist()

# Display as text
print(f'After "{seed_text}", the model\'s top 10 predictions are:\n')
for i, (word, prob) in enumerate(zip(top_words, top_probs_list), 1):
    bar = "#" * int(prob * 100)
    print(f"  {i:2d}. {word:15s} {prob:6.1%}  {bar}")

print(f"\nThe remaining {VOCAB_SIZE - 10:,} words share the other {1 - sum(top_probs_list):.1%} of probability.")

Now let's visualize the same information as a chart. This makes it much easier to see how the probabilities are distributed:

In [ ]:
# Visualize the probability distribution as a bar chart
fig = plot_next_word_probabilities(
    top_probs_list,
    top_words,
    title=f'After "{seed_text}", what word comes next?'
)
plt.show()

print("The top word is the one greedy decoding always picks.")
print("But look at how many other words have a reasonable chance!")
print("By only ever picking the top word, we miss all of them.")

## Strategy 1: Temperature

Temperature is like a **confidence dial** for the AI.

- **Low temperature (0.3)** = the model is very **confident and conservative**. It strongly prefers the top few words and rarely picks anything unexpected. Good for factual, predictable text.
- **Temperature 1.0** = the model uses its learned probabilities **as-is**. This is the "neutral" setting.
- **High temperature (1.5)** = the model is **adventurous and unpredictable**. It flattens the probabilities so that less likely words get a better chance. Good for creative text, but can produce nonsense if too high.

Think of it like cooking: low temperature gives you a safe, predictable dish. High temperature is experimental -- sometimes brilliant, sometimes a mess.

Mathematically, temperature divides the raw scores before converting them to probabilities. Higher values make all options more equal; lower values make the top option even more dominant.

Let's see how the SAME probability distribution changes at different temperatures:

In [ ]:
# Show how temperature changes the probability distribution
temperatures = [0.3, 0.7, 1.0, 1.5]

# Use the same logits from our "Amore mio" example
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, temp in zip(axes, temperatures):
    # Apply temperature: divide logits by temperature, then softmax
    scaled_logits = next_word_logits / temp
    probs = F.softmax(scaled_logits, dim=-1)
    
    # Get top 10
    top_p, top_i = torch.topk(probs, 10)
    words = [tokenizer.decode([idx]) for idx in top_i]
    values = top_p.tolist()
    
    # Plot horizontal bars
    y_pos = np.arange(len(words))
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(words)))
    ax.barh(y_pos, values, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(words, fontsize=9)
    ax.set_xlim(0, max(values) * 1.3)
    ax.invert_yaxis()
    
    # Label
    if temp == 0.3:
        label = f"Temperature = {temp}\n(Very conservative)"
    elif temp == 0.7:
        label = f"Temperature = {temp}\n(Balanced)"
    elif temp == 1.0:
        label = f"Temperature = {temp}\n(Neutral/default)"
    else:
        label = f"Temperature = {temp}\n(Very adventurous)"
    ax.set_title(label, fontsize=11, fontweight='bold')
    
    # Add percentage labels
    for bar, prob in zip(ax.patches, values):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
                f'{prob:.0%}', va='center', fontsize=8)

plt.suptitle(f'How temperature changes predictions after "{seed_text}"',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Notice how:")
print("  - At 0.3, the top word dominates (almost certain to be picked)")
print("  - At 1.5, the probabilities are spread out (many words have a chance)")
print("  - This is why temperature is often called the 'creativity' setting")

Now let's see how temperature affects the actual generated text. We will use the same seed phrase at each temperature:

In [ ]:
# Generate text at different temperatures
seed = "Amore mio"
temperatures = [0.3, 0.7, 1.0, 1.5]

print(f'Generating from "{seed}" at different temperatures:')
print("=" * 60)

for temp in temperatures:
    output = generate_rnn_lstm(
        rnn_model, tokenizer, seed,
        max_length=60, temperature=temp,
        top_k=0, top_p=1.0  # No top-k/top-p filtering yet, just temperature
    )
    label = {0.3: "Conservative", 0.7: "Balanced", 1.0: "Neutral", 1.5: "Adventurous"}
    print(f"\nTemperature = {temp} ({label[temp]}):")
    print(f"  {output}")
    print("-" * 60)

print("\nLow temperature = more repetitive but 'safer'")
print("High temperature = more varied but can become nonsensical")

## Strategy 2: Top-k Sampling

Temperature alone has a problem: even at moderate temperatures, the model COULD pick any of its 50,257 vocabulary words -- including truly bizarre ones that would make no sense. It is unlikely, but it can happen.

**Top-k sampling** solves this by putting a hard limit: **only consider the top k most likely words, and ignore everything else.**

- **top_k = 5**: Only the 5 most likely words are considered. Very focused, but might miss good options.
- **top_k = 50**: The top 50 words are considered. A good balance for most cases.
- **top_k = 500**: The top 500 words are considered. Very permissive -- allows unusual but not impossible choices.

Instead of considering all 50,257 possible next words, only consider the top k most likely ones. This prevents the model from picking very unlikely words (gibberish) while still allowing variety.

In [ ]:
# Generate text with different top-k values
seed = "Amore mio"
top_k_values = [5, 50, 500]

print(f'Generating from "{seed}" with different top-k values:')
print("(Temperature fixed at 0.7 for all)")
print("=" * 60)

for k in top_k_values:
    output = generate_rnn_lstm(
        rnn_model, tokenizer, seed,
        max_length=60, temperature=0.7,
        top_k=k, top_p=1.0  # top_p=1.0 means no top-p filtering
    )
    print(f"\ntop_k = {k} (consider only the {k} most likely words):")
    print(f"  {output}")
    print("-" * 60)

print("\nSmall k = more focused, repetitive (like greedy with slight variety)")
print("Large k = more creative, but still avoids truly random words")

## Strategy 3: Top-p (Nucleus) Sampling

Top-p is **smarter than top-k**. Here is why:

With top-k = 50, you ALWAYS consider exactly 50 words, no matter what. But sometimes the model is very confident and the top 3 words cover 95% of the probability -- in that case, considering 50 words adds unnecessary noise. Other times, the model is uncertain and the top 50 words only cover 60% of the probability -- in that case, we might be cutting off good options.

**Top-p adapts to the situation.** Instead of a fixed number of words, it considers the **smallest set of words whose combined probability exceeds p**.

- **top_p = 0.5**: Consider words until you have covered 50% of the probability. Very focused.
- **top_p = 0.9**: Consider words until you have covered 90% of the probability. The standard choice.
- **top_p = 0.95**: Consider words until you have covered 95%. Slightly more permissive.

Sometimes that set is 3 words, sometimes 100 -- it adapts to the context. This is why top-p is also called "nucleus" sampling: it finds the "nucleus" of likely words.

In [ ]:
# Generate text with different top-p values
seed = "Amore mio"
top_p_values = [0.5, 0.9, 0.95]

print(f'Generating from "{seed}" with different top-p values:')
print("(Temperature fixed at 0.7, no top-k filtering)")
print("=" * 60)

for p in top_p_values:
    output = generate_rnn_lstm(
        rnn_model, tokenizer, seed,
        max_length=60, temperature=0.7,
        top_k=0, top_p=p  # top_k=0 means no top-k filtering
    )
    print(f"\ntop_p = {p} (consider words covering {p:.0%} of probability):")
    print(f"  {output}")
    print("-" * 60)

print("\nTop-p adapts automatically:")
print("  - When the model is confident, it considers fewer words")
print("  - When the model is uncertain, it considers more words")
print("  - This makes it smarter than a fixed top-k value")

## Strategy 4: Repetition Penalty

For Transformer models (like GPT-2), we have one more powerful tool: **repetition penalty**. This directly attacks the repetition problem at its source.

Here is how it works: each time a word appears in the generated text, its probability gets **reduced** for the next position. The more times a word has appeared, the less likely it is to be chosen again.

- **repetition_penalty = 1.0**: No penalty. Words can repeat freely.
- **repetition_penalty = 1.2**: Moderate penalty. Repeated words are 20% less likely. This is the standard choice.
- **repetition_penalty = 1.5**: Strong penalty. Repeated words are 50% less likely. Can sometimes force the model to use awkward alternatives.

To demonstrate this, we need a Transformer model. Let's train one quickly (this will take a few minutes since Transformers are larger and slower):

In [ ]:
# Train a Transformer (GPT-2) model
# This takes longer than RNN because GPT-2 has ~124 million parameters!
print("Building Transformer (GPT-2) model...")
transformer_model = create_transformer_model(tokenizer).to(device)

transformer_optimizer = get_optimizer(transformer_model, is_transformer=True)

print("Training Transformer (this may take 5-10 minutes on CPU)...")
transformer_history = train_model(
    transformer_model, dataloader, transformer_optimizer, device,
    epochs=2, is_transformer=True, model_name="Transformer"
)

print("\nTransformer training complete!")

Now let's see how repetition penalty changes the Transformer's output. We will use the same seed phrase and compare no penalty, moderate penalty, and strong penalty:

In [ ]:
# Compare repetition penalty settings on the Transformer
seed = "Amore mio"
penalties = [1.0, 1.2, 1.5]

print(f'Generating from "{seed}" with different repetition penalties:')
print("(Using Transformer model with temperature=0.7, top_k=50, top_p=0.9)")
print("=" * 60)

for penalty in penalties:
    output = generate_transformer(
        transformer_model, tokenizer, seed,
        max_length=80, temperature=0.7,
        top_k=50, top_p=0.9,
        repetition_penalty=penalty
    )
    label = {1.0: "No penalty", 1.2: "Moderate penalty (recommended)", 1.5: "Strong penalty"}
    print(f"\nrepetition_penalty = {penalty} ({label[penalty]}):")
    print(f"  {output}")
    print("-" * 60)

print("\nWith penalty = 1.0, you may see repeated phrases.")
print("With penalty = 1.2, repetition is reduced while keeping natural flow.")
print("With penalty = 1.5, repetition is strongly discouraged -- sometimes")
print("the model uses unusual words to avoid repeating itself.")

## The Best Recipe: Combining All Strategies

Each strategy on its own helps, but the real magic happens when we **combine** them. The standard "recipe" used by most modern AI systems is:

- **Temperature = 0.7** -- slightly conservative, avoids nonsense
- **Top-k = 50** -- safety net against truly random words
- **Top-p = 0.9** -- smart adaptive filtering
- **Repetition penalty = 1.2** -- gently discourages repetition (Transformer only)

Let's train all three model types and generate with these optimal settings. We also need an LSTM model to complete our comparison:

In [ ]:
# Train an LSTM model (we already have the RNN and Transformer)
print("Training LSTM model...")
lstm_model = LSTMModel(vocab_size=VOCAB_SIZE, embedding_dim=256, hidden_dim=512).to(device)
lstm_optimizer = get_optimizer(lstm_model, is_transformer=False)

lstm_history = train_model(
    lstm_model, dataloader, lstm_optimizer, device,
    epochs=2, is_transformer=False, model_name="LSTM"
)

print("\nAll three models are now trained!")

Now let's generate from all three models using the optimal settings and compare the results side by side:

In [ ]:
# Generate from all three models with optimal settings
seed = "Amore mio"

# Optimal settings
optimal_temp = 0.7
optimal_top_k = 50
optimal_top_p = 0.9
optimal_rep_penalty = 1.2

# Generate from each model
rnn_output = generate_rnn_lstm(
    rnn_model, tokenizer, seed, max_length=80,
    temperature=optimal_temp, top_k=optimal_top_k, top_p=optimal_top_p
)

lstm_output = generate_rnn_lstm(
    lstm_model, tokenizer, seed, max_length=80,
    temperature=optimal_temp, top_k=optimal_top_k, top_p=optimal_top_p
)

transformer_output = generate_transformer(
    transformer_model, tokenizer, seed, max_length=80,
    temperature=optimal_temp, top_k=optimal_top_k, top_p=optimal_top_p,
    repetition_penalty=optimal_rep_penalty
)

# Display comparison
results = {
    "RNN (temperature=0.7, top_k=50, top_p=0.9)": rnn_output,
    "LSTM (temperature=0.7, top_k=50, top_p=0.9)": lstm_output,
    "Transformer (+ repetition_penalty=1.2)": transformer_output,
}

display_generation_comparison(results, title="All Models with Optimal Settings")

print("\nCompare this to the greedy output at the top of the notebook!")
print("The same models, with the same training, produce much better")
print("output when we use smart sampling strategies.")

## Real-World Connection

When you use **ChatGPT**, **Claude**, or any other modern AI system, **these exact same techniques are running behind the scenes.** The "temperature" setting that some AI tools expose is literally the same parameter we explored above.

Here is how it works in practice:

- When you ask an AI for a **factual answer** (like "What is the capital of Italy?"), the system uses a **low temperature** so it picks the most confident response. You want "Rome," not a creative alternative.

- When you ask an AI to **write a creative story**, the system uses a **higher temperature** with top-p sampling so the output is varied and interesting. You want surprise, not the most predictable sentence.

- When you ask an AI to **write a long essay**, repetition penalty is essential. Without it, the AI might repeat the same paragraph structure or even the same sentences over and over.

The difference between a "helpful" AI response and a "creative" AI response is largely in these sampling settings. The underlying model is the same -- it is the generation strategy that changes.

**You now understand one of the most important practical techniques in all of modern AI.**

## Try It Yourself

Now it is your turn! Adjust the settings below and run the cell to see how the output changes. Try these experiments:

1. Set temperature to **0.1** -- what happens? (Very repetitive, like greedy)
2. Set temperature to **2.0** -- what happens? (Mostly nonsense)
3. Set top_k to **3** -- what happens? (Very limited vocabulary)
4. Set top_p to **0.5** -- what happens? (More focused)
5. Try different seed phrases -- Italian phrases work best since the model was trained on Italian lyrics

Change the values in the cell below and re-run it as many times as you like:

In [ ]:
# ============================================================
#   CHANGE THESE VALUES AND RE-RUN THE CELL!
# ============================================================

# Your starting phrase (try Italian phrases!)
my_seed = "La notte"

# Temperature: controls randomness (0.1 = very safe, 2.0 = very wild)
my_temperature = 0.7

# Top-k: how many top words to consider (0 = no limit, 5 = very few, 500 = many)
my_top_k = 50

# Top-p: probability threshold (0.5 = focused, 0.9 = standard, 1.0 = no filter)
my_top_p = 0.9

# ============================================================
#   DO NOT CHANGE BELOW THIS LINE
# ============================================================

print(f"Seed: \"{my_seed}\"")
print(f"Settings: temperature={my_temperature}, top_k={my_top_k}, top_p={my_top_p}")
print("=" * 60)

# Generate from RNN
print("\n[RNN]")
rnn_out = generate_rnn_lstm(
    rnn_model, tokenizer, my_seed, max_length=80,
    temperature=my_temperature, top_k=my_top_k, top_p=my_top_p
)
print(f"  {rnn_out}")

# Generate from LSTM
print("\n[LSTM]")
lstm_out = generate_rnn_lstm(
    lstm_model, tokenizer, my_seed, max_length=80,
    temperature=my_temperature, top_k=my_top_k, top_p=my_top_p
)
print(f"  {lstm_out}")

# Generate from Transformer (with repetition penalty)
print("\n[Transformer]")
trans_out = generate_transformer(
    transformer_model, tokenizer, my_seed, max_length=80,
    temperature=my_temperature, top_k=my_top_k, top_p=my_top_p,
    repetition_penalty=1.2
)
print(f"  {trans_out}")

print("\n" + "=" * 60)
print("Try changing the settings above and running again!")

## Key Takeaways

Here is everything we learned in this notebook, summarized:

1. **Always picking the top word = repetition.** Greedy decoding sounds logical but produces "sole sole sole sole..." because the model gets stuck in loops.

2. **Temperature controls randomness.** Low temperature (0.3) makes the model conservative and predictable. High temperature (1.5) makes it creative but potentially nonsensical. The sweet spot is usually around 0.7.

3. **Top-k limits vocabulary to likely words.** Instead of all 50,257 possible words, only consider the top k. This prevents truly random gibberish while still allowing variety.

4. **Top-p adapts to context.** Unlike top-k which always considers a fixed number of words, top-p considers more words when the model is uncertain and fewer when it is confident. This is smarter and more flexible.

5. **Repetition penalty prevents loops.** For Transformer models, we can directly reduce the probability of words that have already appeared, breaking the repetition cycle.

6. **These techniques are used in all modern AI.** ChatGPT, Claude, and every other AI chatbot use temperature, top-p, and repetition penalty. When you see a "creativity" or "temperature" slider in an AI tool, it is exactly what we explored here.

7. **The model itself does not change.** All these strategies modify how we *choose* from the model's predictions. The model produces the same probabilities every time -- we just pick differently.

## What's Next?

We have built, trained, and improved three AI models. We started with a simple RNN, graduated to an LSTM with better memory, powered up to a GPT-2 Transformer, and now we have learned how to make all of them generate better text using temperature, top-k, top-p, and repetition penalty.

In our **final notebook (Notebook 7)**, we will step back and ask the bigger questions:

- **What are the limitations of AI?** What can these models NOT do, no matter how good our sampling strategies are?
- **What biases does AI carry?** If the training data has biases, does the model inherit them?
- **What are the real opportunities?** Where is this technology heading, and what does it mean for the future?

We have spent six notebooks building technical understanding. Now it is time to think about what it all means.

---

**Notebook 6 of 7 complete.**